<div align="center">
    <br>
    <img alt="logo" src="https://avatars.githubusercontent.com/u/194089448?s=200&v=4" width="100"/>
    <p>
    Build a RAG Chatbot with Open-Weight Models
    </p>
    <p align="center">
        <a href="https://principled-intelligence.com">Principled Intelligence
        </a>
    </p>
    <hr/>
</div>

## Build a RAG Chatbot with Open-Weight Models

This notebook guides you through setting up a RAG chatbot over any document collection using open-weight models running locally. No API keys needed — everything runs on-device via Ollama.

### What you will have at the end

- A locally running RAG chatbot powered by an open-weight model (Qwen, Llama, Gemma, Mistral, …).
- A public URL you can share or point any testing tool at.
- A reproducible setup you can tweak. Swap the docs, the model, or the system prompt.

### Runtime

Go to `Runtime → Change runtime type → T4 GPU` in Colab before running this notebook.

example-rag.svg

### Step 1 — Load Colab secrets

Pull API keys from Colab Secrets into environment variables. `simple-chatbot` and `litellm` read these from the environment at startup, so this cell must run **before** the server is launched.

Only `NGROK_TOKEN` (for the tunnel) are required. No LLM or embedding API key needed — everything runs locally via Ollama.

In [1]:
from google.colab import userdata
import os

#ngrok is needed to expose the RAG chatbot publicly.
ngrok_token = userdata.get("NGROK_TOKEN")


### Step 2 — Install dependencies

Installs:

- **`pciutils` + `zstd`** — system packages required by Ollama (GPU detection and archive decompression).
- **Ollama binary** — pulled from `ollama.com/install.sh` (~80 MB first run).
- `simple-chatbot` — the RAG server (currently installed from a private GitHub branch, hence `GITHUB_TOKEN`).
- `pyngrok` — Python wrapper around the ngrok tunnel binary.
- `datasets` — HuggingFace streaming client for the knowledge base.
- Pinned `numpy` / `pyarrow` versions to avoid a known ABI clash with `datasets`.

First run takes a few minutes.

In [2]:
import os

# Install system dependencies for GPU detection
!sudo apt update
!sudo apt install -y pciutils zstd

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

print("Installing dependencies...")
# Install simple-chatbot and dependencies
!pip install --force-reinstall -q "numpy>=2.0,<2.1" "pyarrow>=15" "datasets>=2.20" \
  git+https://github.com/Principled-Intelligence/simple-chatbot.git \
  pyngrok requests python-dotenv > /dev/null 2>&1
print("Done!")

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 2s (2,094 B/s)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
51 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of

### Step 3 — Configure your chatbot

This is the only cell you usually need to edit. Defaults: stream the first **300** documents from English Wikipedia, use `gemma4:e4b` as the chat model, set a generic assistant system prompt.

**Swap the knowledge base:** point `HF_DATASET` / `HF_CONFIG` at any HuggingFace dataset with a text column, then update `TEXT_FIELD` / `TITLE_FIELD` to match its schema. A commented-out alternative config for the **ACL Anthology** dataset is included.

**Swap the chat model:** set `OLLAMA_MODEL` to any tag Ollama can pull (`qwen3:4b`, `llama3.1:8b`, `mistral:7b`, …). `CHAT_MODEL` is automatically set to `ollama_chat/{OLLAMA_MODEL}` so `litellm` routes completions to the local Ollama server.

**Port separation:** `OLLAMA_PORT` (`11434`) and `CHATBOT_PORT` (`15077`) are kept distinct — ngrok tunnels only the chatbot; Ollama stays private on `localhost`.

**Swap the system prompt:** rewriting `SYSTEM_PROMPT` changes the chatbot's persona and rules.



In [3]:

# Wikipedia Dataset config
HF_DATASET    = "wikimedia/wikipedia"
HF_CONFIG     = "20231101.en"   # required for wikimedia/wikipedia, set it to None if unnecessary for your selected dataset.
TEXT_FIELD    = "text"          # mandatory: column that contains the document body
TITLE_FIELD   = "title"         # optional: column used for filenames; set to None to use row index
HF_SPLIT      = "train"
KB_SIZE = 300

'''
Uncomment this if you want to set up an ACL Anthology assistant.
# ACL Anthology Dataset config
HF_DATASET = "KRLabsOrg/acl-anthology-md"
HF_CONFIG = "fulltext"
HF_SPLIT = "train"
TEXT_FIELD = "markdown"
TITLE_FIELD = None
KB_SIZE = 300
'''



'\nUncomment this if you want to set up an ACL Anthology assistant.\n# ACL Anthology Dataset config\nHF_DATASET = "KRLabsOrg/acl-anthology-md"\nHF_CONFIG = "fulltext"\nHF_SPLIT = "train"\nTEXT_FIELD = "markdown"\nTITLE_FIELD = None\nKB_SIZE = 300\n'

In [4]:
# Model configuration
OLLAMA_MODEL    = "gemma4:e4b"               # Ollama model name
OLLAMA_PORT     = 11434
CHAT_MODEL      = f"ollama_chat/{OLLAMA_MODEL}" # litellm model string for simple-chatbot
EMBEDDING_MODEL = "ollama/mxbai-embed-large"
CHATBOT_PORT    = 15077
SYSTEM_PROMPT   = "You are a Wikipedia expert. Reply to user's questions based on your provided context. If the provided context does not contain the answer please use your prior knowledge."

HF_NAME = HF_DATASET.split("/")[-1]
KB_DIR = f"kbs/{HF_NAME}"

### Step 4 — Download the knowledge base

Streams `KB_SIZE` articles from HuggingFace and writes each one as a `.txt` file under `kbs/<dataset-name>/`. Streaming avoids a full dataset download — we only pay for the bytes we read.

`simple-chatbot` picks up this folder directly via `--docs-dir` and indexes it on startup. Supported file types: `.txt`, `.md`, `.pdf`, `.docx`.

In [5]:
from datasets import load_dataset
import os

os.makedirs(KB_DIR, exist_ok=True)

print(f"Downloading first {KB_SIZE} documents from {HF_DATASET} (split={HF_SPLIT})...")

load_kwargs = dict(split=HF_SPLIT, streaming=True)
if HF_CONFIG is not None:
    dataset = load_dataset(HF_DATASET, HF_CONFIG, **load_kwargs)
else:
    dataset = load_dataset(HF_DATASET, **load_kwargs)

written = 0
for idx, row in enumerate(dataset):
    if written >= KB_SIZE:
        break
    filename = str(row[TITLE_FIELD]).replace("/", "_").replace("\\", "_")[:100] if TITLE_FIELD is not None else f"doc_{idx}"
    body = row[TEXT_FIELD]
    with open(f"{KB_DIR}/{filename}.txt", "w") as f:
        f.write(body)
    written += 1

print(f"✓ Wrote {written} documents to {KB_DIR}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

✓ Wrote 300 documents to kbs/wikipedia


### Step 5 — Start Ollama and pull the model

Three cells set up the local model server:

1. **`ollama serve`** — launched in the background. Logs to `/tmp/ollama.log`.
2. **Health-check loop** — polls `GET /api/tags` until Ollama responds (up to 60 s). If this fails, check the log.
3. **`ollama pull <model>`** — downloads the chat model weights. On first run expect 3–10 minutes; subsequent runs are instant (weights cached on disk).
4. **`ollama pull <embedding-model>`** — downloads the embedding model.

In [6]:
!nohup ollama serve > /tmp/ollama.log 2>&1 &

In [7]:
import requests
import time

print(f"Waiting for Ollama to be ready...")
for attempt in range(30):  # 30 attempts × 2s = 60s max
    try:
        r = requests.get(f"http://localhost:{OLLAMA_PORT}/api/tags", timeout=2)
        if r.status_code == 200:
            print("✓ Ollama is ready")
            break
    except Exception:
        pass
    if attempt % 10 == 0:
        print(f"  ... waiting ({attempt * 2} seconds elapsed)")
    time.sleep(2)
else:
    print("✗ Ollama failed to start — check /tmp/ollama.log")
    raise RuntimeError("Ollama health check failed")

Waiting for Ollama to be ready...
✓ Ollama is ready


In [8]:
import subprocess

print(f"Pulling {OLLAMA_MODEL} (this may take a few minutes on first run)...")
result = subprocess.run(["ollama", "pull", OLLAMA_MODEL], capture_output=True, text=True)
if result.returncode == 0:
    print(f"✓ {OLLAMA_MODEL} is ready")
else:
    print(f"✗ Failed to pull model: {result.stderr}")
    raise RuntimeError(f"ollama pull failed")

Pulling gemma4:e4b (this may take a few minutes on first run)...
✓ gemma4:e4b is ready


In [9]:

print(f"Pulling {EMBEDDING_MODEL}...")
result = subprocess.run(["ollama", "pull", EMBEDDING_MODEL.split("/")[-1]], capture_output=True, text=True)
if result.returncode == 0:
    print(f"✓ {EMBEDDING_MODEL} is ready")
else:
    print(f"✗ Failed to pull {EMBEDDING_MODEL}: {result.stderr}")
    raise RuntimeError("ollama pull all-minilm failed")


Pulling ollama/mxbai-embed-large...
✓ ollama/mxbai-embed-large is ready


### Step 6 — Start the RAG chatbot server

Launches `simple-chatbot serve` as a background process. On startup it:

1. Scans `kbs/<dataset-name>/` for documents.
2. Chunks them (default: 500 tokens, 50-token overlap).
3. Embeds every chunk via the embedding model running locally in Ollama (no external API needed).
4. Stores the vectors in a local ChromaDB collection.
5. Binds a FastAPI server to `localhost:CHATBOT_PORT` exposing `POST /v1/chat/completions`.

Key flags:
- `--chat-api-base http://localhost:OLLAMA_PORT` — routes chat completions to the local Ollama server.
- `--embedding-model <embedding-model>` and `--embedding-api-base http://localhost:OLLAMA_PORT` — routes embeddings to the same local Ollama server.
- `--top-k` <int>— larger retrieval window to give the model more context.



In [10]:
import time

!nohup simple-chatbot serve \
  --docs-dir {KB_DIR} \
  --chat-model "{CHAT_MODEL}" \
  --chat-api-base "http://localhost:{OLLAMA_PORT}" \
  --top-k 20 \
  --embedding-model "{EMBEDDING_MODEL}" \
  --embedding-api-base "http://localhost:{OLLAMA_PORT}" \
  --system-prompt "{SYSTEM_PROMPT}" \
  --port {CHATBOT_PORT} \
  --log-level INFO \
  > /tmp/chatbot.log 2>&1 &


last = ""
while True:
    log = open("/tmp/chatbot.log").read()
    if "Starting HTTP server" in log:
        print("✓ Ready!")
        break
    last_line = log.strip().splitlines()[-1] if log.strip() else ""
    if last_line != last:
        print(last_line)
        last = last_line
    time.sleep(3)

2026-05-14 10:17:35.607 | INFO     | simple_chatbot.indexer:__init__:72 | collection=simple_chatbot persist_dir=/content/.chroma — Initialising ChromaDB
2026-05-14 10:17:39.561 | INFO     | simple_chatbot.indexer:index:197 | batch_count=39 chunk_count=19048 concurrency=10 size=500 — Embedding chunk batches
2026-05-14 10:22:37.359 | INFO     | simple_chatbot.indexer:index:216 | batch_count=39 batch_index=3 done=1500 total=19048 — Indexed chunk batch
2026-05-14 10:22:40.523 | INFO     | simple_chatbot.indexer:index:216 | batch_count=39 batch_index=7 done=3500 total=19048 — Indexed chunk batch
2026-05-14 10:22:42.333 | INFO     | simple_chatbot.indexer:index:216 | batch_count=39 batch_index=9 done=4500 total=19048 — Indexed chunk batch
2026-05-14 10:22:45.965 | INFO     | simple_chatbot.indexer:index:216 | batch_count=39 batch_index=13 done=6500 total=19048 — Indexed chunk batch
2026-05-14 10:22:49.197 | INFO     | simple_chatbot.indexer:index:216 | batch_count=39 batch_index=16 done=8000

### Step 7 — Expose the chatbot via ngrok

Colab runtimes have no public IP. [ngrok](https://ngrok.com/) opens a tunnel from a public HTTPS URL to `CHATBOT_PORT`.
Only the chatbot port is tunnelled — Ollama stays private on `localhost` and is never exposed.

In [11]:
from pyngrok import ngrok
import time
ngrok.set_auth_token(ngrok_token)
tunnel = ngrok.connect(CHATBOT_PORT)
PUBLIC_URL = f"{tunnel.public_url}/v1/chat/completions/"
# wait for ngrok tunnel to set up.
time.sleep(5)
print(f"\n✓ Public URL: {PUBLIC_URL}\n")



✓ Public URL: https://durably-husked-scholar.ngrok-free.dev/v1/chat/completions/



### Step 8 — Smoke-test the endpoint

Sends a single chat request through the public URL. If this prints an answer grounded in the KB, the full pipeline is healthy:

`client → ngrok → FastAPI → retrieval → Ollama → response`

**The first request may take minutes** — Ollama has to warm-load the model into GPU memory. Subsequent requests in the same session are fast. The `timeout=600` reflects this cold-load budget.

If this hangs for more than a few minutes, uncomment the next cell and check the Ollama log.

In [18]:
import json
import requests


def send_message(user_message: str):
  print(f"👤 User: {user_message}")
  response = requests.post(
      PUBLIC_URL,
      json={
          "messages": [
              {"role": "user", "content": user_message}
          ]
      },
      timeout=600,
  )
  response.raise_for_status()

  result = response.json()
  assistant_message = result["choices"][0]["message"]["content"]
  print(f"🤖 Assistant: {assistant_message}")


user_message = "When was Alabama founded?"
send_message(user_message)

print("-"*100)

user_message = "Name three of Andy Warhol's most famous works"
send_message(user_message)

print("-"*100)

user_message = "Briefly summarize one of Hercule Poirot's cases"
send_message(user_message)







👤 User: When was Alabama founded?
🤖 Assistant: Based on the provided context, Alabama's official founding as a state was on **December 14, 1819**, when it was admitted as the 22nd state.

It is worth noting a few other significant dates mentioned in the documents, depending on what aspect of "founding" you are referring to:

*   **First European Settlement:** The region's first European settlement was established at Old Mobile by French colonists in **1702**.
*   **Alabama Territory:** The United States Congress created the Alabama Territory on **March 3, 1817**.
----------------------------------------------------------------------------------------------------
👤 User: Name three of Andy Warhol's most famous works
🤖 Assistant: Based on my knowledge of Pop Art and Andy Warhol's monumental career, three of his most famous and iconic works include:

1.  ***Campbell's Soup Cans*** **(1962):** This series is arguably one of the most revolutionary and defining pieces of Pop Art. By replicat

### Step 9 (Optional) - Having troubles? check the simple-server and ollama logs

In [13]:
'''with open('/tmp/ollama.log') as f:
  print(f.read())
'''

time=2026-05-14T10:17:04.183Z level=INFO source=routes.go:1802 msg="server config" env="map[CUDA_VISIBLE_DEVICES: GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: NO_PROXY: OLLAMA_CONTEXT_LENGTH:0 OLLAMA_DEBUG:INFO OLLAMA_DEBUG_LOG_REQUESTS:false OLLAMA_EDITOR: OLLAMA_FLASH_ATTENTION:false OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://127.0.0.1:11434 OLLAMA_KEEP_ALIVE:5m0s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MAX_TRANSFER_STREAMS:4 OLLAMA_MODELS:/root/.ollama/models OLLAMA_MULTIUSER_CACHE:false OLLAMA_NEW_ENGINE:false OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NO_CLOUD:false OLLAMA_NUM_PARALLEL:1 OLLAMA_ORIGINS:[http://localhost https://localhost http://localhost:* https://localhost:* http://127.0.0.1 https://127.0.0.1 http://127.0.0.1:* https://127.0.0.1:* http://0.0.0.0 https://0.0.0.0 http://0.0.0.0:* https://0.0.0.0:* app://* fi

In [14]:
'''with open('/tmp/chatbot.log') as f:
  print(f.read())
'''


2026-05-14 10:17:35.526 | INFO     | simple_chatbot.cli:serve:111 — simple-chatbot starting up
2026-05-14 10:17:35.527 | INFO     | simple_chatbot.cli:serve:118 | chat_model=ollama_chat/gemma4:e4b docs_dir=kbs/wikipedia embedding_model=ollama/mxbai-embed-large log_format=pretty log_level=INFO — Runtime configuration
2026-05-14 10:17:35.527 | INFO     | simple_chatbot.cli:serve:126 | chroma_dir=.chroma chunk_overlap=50 chunk_size=500 collection=simple_chatbot max_tool_rounds=5 top_k=20 — Index configuration
2026-05-14 10:17:35.527 | INFO     | simple_chatbot.cli:serve:128 | api_base=http://localhost:11434 — Chat API base override
2026-05-14 10:17:35.528 | INFO     | simple_chatbot.cli:serve:130 | api_base=http://localhost:11434 — Embedding API base override
2026-05-14 10:17:35.528 | INFO     | simple_chatbot.cli:serve:165 | docs_dir=kbs/wikipedia — Loading documents
2026-05-14 10:17:35.529 | INFO     | simple_chatbot.loader:load_documents:48 | docs_dir=/content/kbs/wikipedia — Scanning 

### Step 9 — (Optional) Export the knowledge base

Zips the `kbs/` folder and downloads it to your local machine.

In [15]:
'''import shutil
from google.colab import files

# Zip the kbs folder
shutil.make_archive('/content/kbs', 'zip', '/content', 'kbs')

# Download
files.download('/content/kbs.zip')
'''

"import shutil\nfrom google.colab import files\n\n# Zip the kbs folder\nshutil.make_archive('/content/kbs', 'zip', '/content', 'kbs')\n\n# Download\nfiles.download('/content/kbs.zip')\n"

### Step 10 — Stop the servers

Stops **both** `simple-chatbot serve` and `ollama serve`, plus the ngrok tunnel.

> **Heads-up:** This cell is commented out so it doesn't run by accident when you hit *Run all*. Uncomment it when you actually want to stop the servers.


In [16]:
'''!pkill -f "simple-chatbot serve"
!pkill -f "ollama serve"
ngrok.kill()
'''

'!pkill -f "simple-chatbot serve"\n!pkill -f "ollama serve"\nngrok.kill()\n'

## Want to test it automatically? Use Spectral!

Your chatbot is live. **Spectral** can probe it at scale by simulating user inputs and scoring the results with no extra code on your side.

architecture.svg

1. Open the [Spectral dashboard](https://spectral.principled.app/).
2. Create a target and paste the public URL from Step 7 as its endpoint.
3. Upload the knowledge base you just built (optional, improves factuality scoring).
4. Define the principles you want the chatbot to respect.
5. Launch an evaluation.

Within a few minutes you'll have a scored report flagging where your chatbot falls short.

### Troubleshooting

- **`ngrok` refuses to start** - `NGROK_TOKEN` is missing or invalid. Generate a fresh one at [dashboard.ngrok.com](https://dashboard.ngrok.com/get-started/your-authtoken), re-run Steps 1 and 7.
- **Smoke test returns 502 or an empty response** - the chatbot is still indexing. Check `/tmp/chatbot.log` (Step 5) and retry once it logs `Starting HTTP server`.
- **Public URL stopped working** - the Colab runtime was recycled. Re-run Step 7 and update the URL. For longer runs, use [spectral-bridge](https://github.com/Principled-Intelligence/spectral-bridge).
